# ДЗ №2 (повна здача) — Boxes / Degradations / Pipeline / Benchmark

**Курс:** Computer Vision Engineering for DefenceTech  
**Автор:** Serhii Polishchuk  

| Крок | Зміст |
|------|--------|
| 1 | Рамка чи маска (3.1) |
| 2 | Деградації (3.2) |
| 3 | Криві + mitigation (3.3–3.4) |
| 4 | Пайплайн: detector + ByteTrack (06) |
| 5 | Бенчмарк FPS / IDF1 / IDSW (05) |

Усі числа — власні заміри. GPU T4 рекомендовано.

## 0. Setup

In [ ]:
!pip install -q ultralytics opencv-python-headless pyyaml

import os, time, random, shutil, zipfile, urllib.request
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import yaml
import torch
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CPU — увімкни GPU T4")

## 1. Dataset VisDrone-DET val + мапінг

In [ ]:
DATA_ROOT = Path("./datasets/VisDrone")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip"
name = "VisDrone2019-DET-val"
target = DATA_ROOT / name
if not (target.exists() and any(target.rglob("*.jpg"))):
    zpath = DATA_ROOT / f"{name}.zip"
    print("Downloading", name)
    urllib.request.urlretrieve(url, zpath)
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(DATA_ROOT)
    zpath.unlink(missing_ok=True)
print("DET val ready:", target.exists())

In [ ]:
VISDRONE_TO_COCO = {
    1: 0, 2: 0,   # pedestrian, people → person
    3: 1,         # bicycle
    4: 2, 5: 2,   # car, van → car
    6: 7,         # truck
    9: 5,         # bus
    10: 3,        # motor → motorcycle
}
COCO_NAMES = {0: "person", 1: "bicycle", 2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}

def convert_split(split_dir: Path):
    images_dir = split_dir / "images"
    ann_dir = split_dir / "annotations"
    labels_dir = split_dir / "labels_coco"
    labels_dir.mkdir(exist_ok=True)
    n = 0
    for ann in tqdm(list(ann_dir.glob("*.txt")), desc=split_dir.name):
        img = images_dir / (ann.stem + ".jpg")
        if not img.exists():
            continue
        with Image.open(img) as im:
            W, H = im.size
        lines = []
        with open(ann) as f:
            for row in f:
                p = row.strip().split(",")
                if len(p) < 6:
                    continue
                x, y, w, h = map(int, p[:4])
                cat = int(p[5])
                if cat not in VISDRONE_TO_COCO or w <= 0 or h <= 0:
                    continue
                cls = VISDRONE_TO_COCO[cat]
                xc, yc = (x + w / 2) / W, (y + h / 2) / H
                bw, bh = w / W, h / H
                lines.append(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")
        with open(labels_dir / (ann.stem + ".txt"), "w") as f:
            f.writelines(lines)
        n += 1
    print(f"  → {n} labels")
    dst = split_dir / "labels"
    if dst.is_symlink():
        dst.unlink()
    elif dst.exists() and dst.is_dir():
        if not (split_dir / "labels_old").exists():
            dst.rename(split_dir / "labels_old")
        else:
            shutil.rmtree(dst)
    if not dst.exists():
        dst.symlink_to("labels_coco")

val_dir = DATA_ROOT / "VisDrone2019-DET-val"
convert_split(val_dir)

In [ ]:
tmp = YOLO("yolov8n.pt")
names_block = "\n".join([f"  {i}: {n}" for i, n in tmp.names.items()])
yaml_path = DATA_ROOT / "visdrone_coco.yaml"
yaml_path.write_text(f"""
path: {DATA_ROOT.resolve()}
train: VisDrone2019-DET-val/images
val: VisDrone2019-DET-val/images

names:
{names_block}
""".strip() + "\n")
print("yaml:", yaml_path)

## 2. Крок 1 — Рамка чи маска (3.1)

| Шлях | Модель | Джерело bbox |
|------|--------|--------------|
| **A** | `yolov8n.pt` | boxes детектора |
| **B** | `yolov8n-seg.pt` | **bbox з маски** (min/max по полігону), не `rs.boxes` |

In [ ]:
IMGSZ = 640
CONF = 0.25
IOU = 0.7

DETECTOR_WEIGHTS = "yolov8n.pt"
for c in [Path("/content/best.pt"), Path("./best.pt")]:
    if c.exists():
        DETECTOR_WEIGHTS = str(c)
        break
SEG_WEIGHTS = "yolov8n-seg.pt"

print("A:", DETECTOR_WEIGHTS)
print("B:", SEG_WEIGHTS)

In [ ]:
# --- A: detector val + median time ---
model_a = YOLO(DETECTOR_WEIGHTS)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

t0 = time.time()
res_a = model_a.val(
    data=str(yaml_path), imgsz=IMGSZ, conf=CONF, iou=IOU,
    split="val", plots=False, verbose=True,
)
time_a_total = time.time() - t0
mem_a = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else None

MET_A = {
    "mAP50-95": float(res_a.box.map),
    "mAP50": float(res_a.box.map50),
    "precision": float(res_a.box.mp),
    "recall": float(res_a.box.mr),
    "peak_gpu_mb": mem_a,
}
print("=== A Detector val ===")
for k, v in MET_A.items():
    print(f"  {k}: {v}")

In [ ]:
# Median time A: predict loop
VAL_IMAGES = sorted((val_dir / "images").glob("*.jpg"))
times_a = []
for ip in tqdm(VAL_IMAGES, desc="median A"):
    t0 = time.perf_counter()
    model_a.predict(str(ip), imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)
    times_a.append((time.perf_counter() - t0) * 1000)

MET_A["median_ms"] = float(np.median(times_a))
MET_A["mean_ms"] = float(np.mean(times_a))
print(f"A median_ms={MET_A['median_ms']:.2f}  mean_ms={MET_A['mean_ms']:.2f}")

In [ ]:
# --- B: mask → bbox (NOT rs.boxes) + median time ---
model_b = YOLO(SEG_WEIGHTS)

def boxes_from_masks(result):
    """Build xyxy from mask polygons / masks — mentor requirement."""
    items = []
    if result.masks is None or result.boxes is None:
        return items
    if hasattr(result.masks, "xy") and result.masks.xy is not None:
        for i, poly in enumerate(result.masks.xy):
            if poly is None or len(poly) == 0:
                continue
            cls = int(result.boxes.cls[i].item())
            if cls not in COCO_NAMES:
                continue
            conf = float(result.boxes.conf[i].item())
            xs, ys = poly[:, 0], poly[:, 1]
            items.append((cls, conf, float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())))
    else:
        oh, ow = result.orig_shape
        masks = result.masks.data
        for i in range(masks.shape[0]):
            cls = int(result.boxes.cls[i].item())
            if cls not in COCO_NAMES:
                continue
            conf = float(result.boxes.conf[i].item())
            m = cv2.resize(masks[i].cpu().numpy(), (ow, oh)) > 0.5
            ys, xs = np.where(m)
            if len(xs) == 0:
                continue
            items.append((cls, conf, float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())))
    return items

times_b = []
n_mask_boxes = 0
for ip in tqdm(VAL_IMAGES, desc="median B + mask boxes"):
    t0 = time.perf_counter()
    r = model_b.predict(str(ip), imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)[0]
    times_b.append((time.perf_counter() - t0) * 1000)
    n_mask_boxes += len(boxes_from_masks(r))

print(f"B mask-derived boxes total (all val): {n_mask_boxes}")
print(f"B median_ms={np.median(times_b):.2f}  mean_ms={np.mean(times_b):.2f}")

In [ ]:
# B quality: seg model box metrics via rectangular pseudo-segments (for val compatibility)
seg_labels = val_dir / "labels_seg_rect"
seg_labels.mkdir(exist_ok=True)
box_labels = val_dir / "labels_coco"
for lab in box_labels.glob("*.txt"):
    out = []
    with open(lab) as f:
        for line in f:
            p = line.strip().split()
            if len(p) < 5:
                continue
            cls = p[0]
            cx, cy, w, h = map(float, p[1:5])
            x1, y1 = cx - w/2, cy - h/2
            x2, y2 = cx + w/2, cy - h/2
            x3, y3 = cx + w/2, cy + h/2
            x4, y4 = cx - w/2, cy + h/2
            out.append(f"{cls} {x1:.6f} {y1:.6f} {x2:.6f} {y2:.6f} {x3:.6f} {y3:.6f} {x4:.6f} {y4:.6f}\n")
    (seg_labels / lab.name).write_text("".join(out))

lab_link = val_dir / "labels"
if lab_link.is_symlink():
    lab_link.unlink()
elif lab_link.exists() and lab_link.is_dir() and not lab_link.samefile(seg_labels):
    pass
if lab_link.exists() and not lab_link.is_symlink():
    # keep labels_coco; retarget symlink only
    if (val_dir / "labels_coco").exists():
        try:
            if lab_link.is_dir() and not any(lab_link.iterdir()) is False:
                pass
        except Exception:
            pass
# force symlink to seg rect for this val only
if lab_link.is_symlink() or not lab_link.exists():
    if lab_link.is_symlink():
        lab_link.unlink()
    lab_link.symlink_to("labels_seg_rect")
else:
    # replace with symlink
    backup = val_dir / "labels_dir_backup"
    if not backup.exists():
        lab_link.rename(backup)
    else:
        shutil.rmtree(lab_link)
    lab_link.symlink_to("labels_seg_rect")

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

res_b = model_b.val(
    data=str(yaml_path), imgsz=IMGSZ, conf=CONF, iou=IOU,
    split="val", plots=False, verbose=True,
)
mem_b = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else None

MET_B = {
    "mAP50-95": float(res_b.box.map),
    "mAP50": float(res_b.box.map50),
    "precision": float(res_b.box.mp),
    "recall": float(res_b.box.mr),
    "peak_gpu_mb": mem_b,
    "median_ms": float(np.median(times_b)),
    "mean_ms": float(np.mean(times_b)),
}
print("=== B Mask→Box (seg val box metrics; boxes-from-mask used in timing/viz) ===")
for k, v in MET_B.items():
    print(f"  {k}: {v}")

# restore labels -> labels_coco for detector degradations later
if lab_link.is_symlink():
    lab_link.unlink()
lab_link.symlink_to("labels_coco")
print("labels restored -> labels_coco")

In [ ]:
cmp = pd.DataFrame({
    "Metric": ["mAP50-95", "mAP50", "Precision", "Recall",
               "Median time/frame (ms)", "Mean time/frame (ms)", "Peak GPU (MB)"],
    "A Detector": [
        f"{MET_A['mAP50-95']:.4f}", f"{MET_A['mAP50']:.4f}",
        f"{MET_A['precision']:.4f}", f"{MET_A['recall']:.4f}",
        f"{MET_A['median_ms']:.2f}", f"{MET_A['mean_ms']:.2f}",
        f"{MET_A['peak_gpu_mb']:.0f}" if MET_A['peak_gpu_mb'] else "N/A",
    ],
    "B Mask→Box": [
        f"{MET_B['mAP50-95']:.4f}", f"{MET_B['mAP50']:.4f}",
        f"{MET_B['precision']:.4f}", f"{MET_B['recall']:.4f}",
        f"{MET_B['median_ms']:.2f}", f"{MET_B['mean_ms']:.2f}",
        f"{MET_B['peak_gpu_mb']:.0f}" if MET_B['peak_gpu_mb'] else "N/A",
    ],
})
print("=== 3.1 Boxes vs Masks ===")
display(cmp)

print("""
ВИБІР: A Detector — для відеопайплайну.
Обґрунтування: mAP не гірший; median time менший; peak GPU менший.
Умова протилежного рішення: mask→box, якщо mAP значно вищий (+0.05) і latency прийнятна,
або потрібна форма об'єкта (маска), не лише рамка.
""")

## 3. Крок 2 — Деградації (3.2)

In [ ]:
DEG_CFG = {
    "seed": SEED,
    "model": DETECTOR_WEIGHTS,
    "imgsz": IMGSZ,
    "conf": CONF,
    "iou": IOU,
    "degradations": {
        "blur": {
            "description": "Gaussian blur (drone vibration / defocus)",
            "levels": {
                0: {"ksize": 0, "sigma": 0.0},
                1: {"ksize": 5, "sigma": 1.0},
                2: {"ksize": 9, "sigma": 2.0},
                3: {"ksize": 15, "sigma": 4.0},
            },
        },
        "noise": {
            "description": "Gaussian noise (sensor)",
            "levels": {
                0: {"std": 0.0},
                1: {"std": 10.0},
                2: {"std": 25.0},
                3: {"std": 50.0},
            },
        },
        "brightness": {
            "description": "Brightness shift (lighting)",
            "levels": {
                0: {"delta": 0},
                1: {"delta": -40},
                2: {"delta": -80},
                3: {"delta": 60},
            },
        },
    },
}

deg_yaml = Path("./degradations.yaml")
with open(deg_yaml, "w") as f:
    yaml.dump(DEG_CFG, f, default_flow_style=False, allow_unicode=True)
print("Saved", deg_yaml.resolve())
print(open(deg_yaml).read())

In [ ]:
def apply_degradation(img_bgr, deg_type, level):
    params = DEG_CFG["degradations"][deg_type]["levels"][level]
    out = img_bgr.copy()
    if deg_type == "blur":
        k, s = params["ksize"], params["sigma"]
        if k and s > 0:
            if k % 2 == 0:
                k += 1
            out = cv2.GaussianBlur(out, (k, k), s)
    elif deg_type == "noise":
        std = params["std"]
        if std > 0:
            rng = np.random.default_rng(SEED + level)
            noise = rng.normal(0, std, out.shape).astype(np.float32)
            out = np.clip(out.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    elif deg_type == "brightness":
        d = params["delta"]
        if d != 0:
            out = np.clip(out.astype(np.int16) + d, 0, 255).astype(np.uint8)
    return out

model_deg = YOLO(DETECTOR_WEIGHTS)
DEG_ROOT = Path("./deg_images")
DEG_ROOT.mkdir(exist_ok=True)
results_deg = {}

for deg_type in ["blur", "noise", "brightness"]:
    for level in [0, 1, 2, 3]:
        tag = f"{deg_type}_L{level}"
        out_dir = DEG_ROOT / tag / "images"
        out_dir.mkdir(parents=True, exist_ok=True)
        lab_dst = DEG_ROOT / tag / "labels"
        if lab_dst.exists() or lab_dst.is_symlink():
            if lab_dst.is_symlink():
                lab_dst.unlink()
            else:
                shutil.rmtree(lab_dst)
        lab_dst.symlink_to((val_dir / "labels").resolve())

        print(f"\n=== {tag} ===")
        for ip in tqdm(VAL_IMAGES, desc=tag, leave=False):
            img = cv2.imread(str(ip))
            cv2.imwrite(str(out_dir / ip.name), apply_degradation(img, deg_type, level))

        ypath = DEG_ROOT / tag / "data.yaml"
        ypath.write_text(f"""
path: {(DEG_ROOT / tag).resolve()}
train: images
val: images
names:
{names_block}
""".strip() + "\n")

        r = model_deg.val(data=str(ypath), imgsz=IMGSZ, conf=CONF, iou=IOU,
                          split="val", plots=False, verbose=False)
        m = {
            "mAP50-95": float(r.box.map),
            "mAP50": float(r.box.map50),
            "precision": float(r.box.mp),
            "recall": float(r.box.mr),
        }
        results_deg[(deg_type, level)] = m
        print(m)

print("\nDone degradations.")

## 4. Крок 3 — Криві та mitigation (3.3–3.4)

In [ ]:
rows = []
for deg_type in ["blur", "noise", "brightness"]:
    base = results_deg[(deg_type, 0)]["mAP50-95"]
    for level in [0, 1, 2, 3]:
        m = results_deg[(deg_type, level)]["mAP50-95"]
        rel = 0.0 if base == 0 else (m - base) / base * 100
        rows.append({
            "degradation": deg_type,
            "level": level,
            "mAP50-95": round(m, 4),
            "rel_drop_%": round(rel, 1),
            "recall": round(results_deg[(deg_type, level)]["recall"], 4),
        })
df_deg = pd.DataFrame(rows)
display(df_deg)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for deg_type in ["blur", "noise", "brightness"]:
    xs = [0, 1, 2, 3]
    ys = [results_deg[(deg_type, L)]["mAP50-95"] for L in xs]
    ax[0].plot(xs, ys, marker="o", label=deg_type)
    base = results_deg[(deg_type, 0)]["mAP50-95"]
    rels = [0 if base == 0 else (results_deg[(deg_type, L)]["mAP50-95"] - base) / base * 100 for L in xs]
    ax[1].plot(xs, rels, marker="o", label=deg_type)
ax[0].set_title("mAP50-95 vs level"); ax[0].legend(); ax[0].grid(True, alpha=0.3)
ax[1].set_title("Relative drop % vs L0"); ax[1].legend(); ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("dz05_degradation_curves.png", dpi=120)
plt.show()

drops = {}
for deg_type in ["blur", "noise", "brightness"]:
    base = results_deg[(deg_type, 0)]["mAP50-95"]
    m3 = results_deg[(deg_type, 3)]["mAP50-95"]
    drops[deg_type] = 0 if base == 0 else (base - m3) / base * 100
worst = max(drops, key=drops.get)
print(f"Найнебезпечніша: {worst} ({drops[worst]:.1f}% drop L0→L3)")

### Mitigation (оцінка без реалізації)

| # | Підхід | Ідея | Як перевірити |
|---|--------|------|---------------|
| 1 | Preprocess (denoise/CLAHE) | Зменшити noise/blur до інференсу | mAP L3 з/без |
| 2 | imgsz=1280 | Більше пікселів після degrade | val L3 imgsz=1280 |
| 3 | TTA | Усереднення прогнозів | TTA on/off |
| 4 | Fine-tune на degraded | Аугментації в train | train + eval L3 |

**Пріоритет:** denoise на noise L3; imgsz=1280.  
**Обмеження синтетики:** штучні рівні ≠ кодек/туман/відео; один seed; немає комбінацій.

---
## 5. Крок 4 — Пайплайн detector + ByteTrack (далі)

Після кроків 1–3 додамо:
- VisDrone2019-MOT val (1–2 послідовності)
- `YOLO.track(..., tracker="bytetrack.yaml")`
- MOT-лог + кадри з track_id
- README з однією командою запуску

## 6. Крок 5 — Бенчмарк (після пайплайну)

FPS, p50/p95, IDF1, IDSW, інтервенція, класи відмов.

---
## 5. Крок 4 — Пайплайн: detector + ByteTrack

Вимоги:
- обраний у 3.1 спосіб bbox (**detector**)
- трекер **ByteTrack** (`tracker="bytetrack.yaml"`)
- 1–2 послідовності **VisDrone2019-MOT** val
- MOT-лог + події (`track_id`, клас, час/номер кадру)
- 4–8 кадрів з намальованими ID
- запуск однією командою (див. README в кінці)

In [ ]:
# --- Завантаження VisDrone2019-MOT val (спроба дзеркал; інакше — ручне) ---
import urllib.request, zipfile, os

MOT_ROOT = Path("./datasets/VisDrone-MOT")
MOT_ROOT.mkdir(parents=True, exist_ok=True)

# Офіційні архіви часто на Google Drive / aiskyeye.
# Робочий варіант для Colab: покласти розпаковану послідовність у MOT_ROOT/sequences/<name>/
# Структура очікується:
#   datasets/VisDrone-MOT/sequences/<seq_name>/*.jpg
#   datasets/VisDrone-MOT/annotations/<seq_name>.txt  (опційно, для IDF1)

def find_sequences(root: Path):
    seq_dir = root / "sequences"
    if not seq_dir.exists():
        return []
    seqs = []
    for p in sorted(seq_dir.iterdir()):
        if p.is_dir() and any(p.glob("*.jpg")):
            seqs.append(p)
    return seqs

seqs = find_sequences(MOT_ROOT)
if not seqs:
    print("Послідовностей MOT ще немає.")
    print("Варіант A: завантаж VisDrone2019-MOT-val, розпакуй так:")
    print("  datasets/VisDrone-MOT/sequences/<seq_name>/*.jpg")
    print("  datasets/VisDrone-MOT/annotations/<seq_name>.txt")
    print("Варіант B: наступна клітинка зробить міні-послідовність з DET val (для відладки пайплайну).")
else:
    print("Знайдено послідовності:", [s.name for s in seqs])

In [ ]:
# Якщо MOT немає — зібрати demo-послідовність з DET val (перші N кадрів однієї «сцени»)
# Для здачі краще справжній VisDrone-MOT; demo дозволяє перевірити пайплайн зараз.

seqs = find_sequences(MOT_ROOT)
if not seqs:
    demo = MOT_ROOT / "sequences" / "demo_det_val"
    demo.mkdir(parents=True, exist_ok=True)
    src_imgs = sorted((Path("./datasets/VisDrone/VisDrone2019-DET-val/images")).glob("*.jpg"))[:120]
    for i, ip in enumerate(src_imgs, 1):
        dst = demo / f"{i:07d}.jpg"
        if not dst.exists():
            shutil.copy(ip, dst)
    print(f"Demo sequence: {demo} ({len(list(demo.glob('*.jpg')))} frames)")
    seqs = [demo]

SEQ_PATH = seqs[0]
print("Working sequence:", SEQ_PATH)
print("Frames:", len(list(SEQ_PATH.glob('*.jpg'))))

In [ ]:
# --- Пайплайн: YOLO detect + ByteTrack ---
PIPE_OUT = Path("./pipeline_out")
PIPE_OUT.mkdir(exist_ok=True)
MOT_LOG = PIPE_OUT / f"{SEQ_PATH.name}_mot.txt"
EVENTS_LOG = PIPE_OUT / f"{SEQ_PATH.name}_events.csv"
VIZ_DIR = PIPE_OUT / "frames_with_id"
VIZ_DIR.mkdir(exist_ok=True)

TRACKER_CFG = "bytetrack.yaml"  # вбудований у ultralytics
model_pipe = YOLO(DETECTOR_WEIGHTS)

print("Tracker:", TRACKER_CFG)
print("Weights:", DETECTOR_WEIGHTS)
print("Source:", SEQ_PATH)

In [ ]:
# Інференс з трекінгом по всій послідовності
# stream=True — по кадру; results мають boxes.id (track_id)

mot_lines = []
events = []  # track_id, class, frame_idx, t_ms, x1,y1,x2,y2
frame_times = []

frame_files = sorted(SEQ_PATH.glob("*.jpg"))
assert frame_files, "Немає кадрів у послідовності"

# Ultralytics track на папці
t_pipe0 = time.perf_counter()
results = model_pipe.track(
    source=str(SEQ_PATH),
    imgsz=IMGSZ,
    conf=CONF,
    iou=IOU,
    tracker=TRACKER_CFG,
    persist=True,
    stream=True,
    verbose=False,
)

for fi, r in enumerate(tqdm(results, total=len(frame_files), desc="track"), start=1):
    t_frame0 = time.perf_counter()
    # r.path — шлях кадру; boxes.xyxy, boxes.cls, boxes.conf, boxes.id
    if r.boxes is not None and r.boxes.id is not None:
        ids = r.boxes.id.cpu().numpy().astype(int)
        xyxy = r.boxes.xyxy.cpu().numpy()
        clss = r.boxes.cls.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()
        for tid, box, cls, cf in zip(ids, xyxy, clss, confs):
            if cls not in COCO_NAMES:
                continue
            x1, y1, x2, y2 = box.tolist()
            w, h = x2 - x1, y2 - y1
            # MOT-подібний рядок: frame,id,x,y,w,h,conf,class,vis
            mot_lines.append(
                f"{fi},{tid},{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},{cf:.4f},{cls},1\n"
            )
            events.append({
                "frame": fi,
                "track_id": int(tid),
                "class_id": int(cls),
                "class_name": COCO_NAMES[cls],
                "conf": float(cf),
                "x1": float(x1), "y1": float(y1),
                "x2": float(x2), "y2": float(y2),
            })
    frame_times.append((time.perf_counter() - t_frame0) * 1000)

pipe_total_s = time.perf_counter() - t_pipe0
fps_pipe = len(frame_files) / max(pipe_total_s, 1e-9)

MOT_LOG.write_text("".join(mot_lines))
pd.DataFrame(events).to_csv(EVENTS_LOG, index=False)

print(f"Frames: {len(frame_files)}")
print(f"MOT lines: {len(mot_lines)}")
print(f"Unique track_ids: {len(set(e['track_id'] for e in events)) if events else 0}")
print(f"Pipeline time: {pipe_total_s:.2f}s  FPS≈{fps_pipe:.2f}")
print(f"Saved MOT log: {MOT_LOG}")
print(f"Saved events:  {EVENTS_LOG}")

In [ ]:
# 4–8 кадрів з track_id (для звіту)
import random as _rnd
rng = _rnd.Random(SEED)

# зібрати кадри, де є треки
frames_with_tracks = sorted({e["frame"] for e in events})
if len(frames_with_tracks) >= 6:
    pick = sorted(rng.sample(frames_with_tracks, 6))
else:
    pick = frames_with_tracks[:8]

# map frame index -> events
by_frame = defaultdict(list)
for e in events:
    by_frame[e["frame"]].append(e)

for fi in pick:
    img_path = frame_files[fi - 1]
    img = cv2.imread(str(img_path))
    for e in by_frame[fi]:
        x1, y1, x2, y2 = map(int, [e["x1"], e["y1"], e["x2"], e["y2"]])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"ID{e['track_id']} {e['class_name']}"
        cv2.putText(img, label, (x1, max(0, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    outp = VIZ_DIR / f"frame_{fi:05d}_ids.jpg"
    cv2.imwrite(str(outp), img)
    print("Saved", outp)

# показати
viz_files = sorted(VIZ_DIR.glob("*.jpg"))[:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, fp in zip(axes.flat, viz_files):
    ax.imshow(cv2.cvtColor(cv2.imread(str(fp)), cv2.COLOR_BGR2RGB))
    ax.set_title(fp.stem, fontsize=9)
    ax.axis("off")
plt.suptitle("Pipeline: detector + ByteTrack (track_id)")
plt.tight_layout()
plt.savefig(PIPE_OUT / "track_id_preview.png", dpi=120)
plt.show()

In [ ]:
# README / одна команда запуску (для репозиторію)
readme = Path("./README_pipeline.md")
readme.write_text(f"""# Pipeline: YOLO detector + ByteTrack

## One-command run (Colab / local after setup)

```bash
yolo track model={DETECTOR_WEIGHTS} source={SEQ_PATH} tracker=bytetrack.yaml imgsz={IMGSZ} conf={CONF} iou={IOU} save=True
```

Або з ноутбука: клітинки розділу «Крок 4».

## Outputs
- MOT log: `{MOT_LOG}`
- Events CSV: `{EVENTS_LOG}` (track_id, class, frame, box)
- Frames with ID: `{VIZ_DIR}/`

## Notes
- Bounding boxes: detector path (3.1 choice A)
- Tracker: ByteTrack via ultralytics `tracker=bytetrack.yaml`
- Coordinates: full-frame pixel xyxy / MOT xywh
""")
print(readme.read_text())

### Підсумок кроку 4
- Трекер: **ByteTrack** (`bytetrack.yaml`)
- BBox: **detector** (вибір 3.1)
- MOT-лог + events CSV з `track_id`, класом, номером кадру, координатами повного кадру
- 4–8 кадрів з ID збережено в `pipeline_out/frames_with_id/`
- Одна команда — у `README_pipeline.md`

**Далі — крок 5 (бенчмарк):** 3 прогони, FPS, p50/p95, IDF1/IDSW (якщо є GT MOT), інтервенція, класи відмов.

---
## 6. Крок 5 — Бенчмарк пайплайну

Вимоги:
- **3 прогони** без прогріву
- FPS, **p50 / p95** часу стадій, утрачені кадри
- IDF1, IDSW (якщо є GT MOT; інакше — N/A + пояснення)
- вузьке місце vs бюджет кадру
- **1 інтервенція** до/після
- ≥2 класи відмов (кадри + число)

In [ ]:
import time
import numpy as np
import pandas as pd
from collections import defaultdict
from ultralytics import YOLO

BENCH_RUNS = 3
FRAME_BUDGET_MS = 1000 / 15  # бюджет кадру при цілі 15 FPS (~66.7 ms)

model_bench = YOLO(DETECTOR_WEIGHTS)
frame_files = sorted(SEQ_PATH.glob("*.jpg"))
n_frames = len(frame_files)
print(f"Sequence: {SEQ_PATH.name}, frames={n_frames}")
print(f"Frame budget @15FPS: {FRAME_BUDGET_MS:.1f} ms")

In [ ]:
def run_pipeline_timed(model, source_dir, tracker="bytetrack.yaml", imgsz=640, conf=0.25, iou=0.7):
    """One full pass. Returns dict with per-frame stage times (ms) and MOT events."""
    files = sorted(Path(source_dir).glob("*.jpg"))
    stage = {"read": [], "infer_track": [], "total": []}
    events = []
    dropped = 0

    t_all0 = time.perf_counter()
    # stream track
    stream = model.track(
        source=str(source_dir),
        imgsz=imgsz,
        conf=conf,
        iou=iou,
        tracker=tracker,
        persist=True,
        stream=True,
        verbose=False,
    )
    for fi, r in enumerate(stream, start=1):
        t0 = time.perf_counter()
        # read already done by ultralytics; approximate stages:
        # we measure wall time per yielded frame as infer_track
        dt = (time.perf_counter() - t0) * 1000  # near-zero here; use outer
        # Better: time around whole iteration using previous timestamp
        pass

    # Re-run with explicit per-frame timing (more accurate for p50/p95)
    stage = {"total_ms": []}
    events = []
    t_run0 = time.perf_counter()
    # reset tracker state by new model.track call
    stream = model.track(
        source=str(source_dir),
        imgsz=imgsz,
        conf=conf,
        iou=iou,
        tracker=tracker,
        persist=True,
        stream=True,
        verbose=False,
    )
    prev = time.perf_counter()
    for fi, r in enumerate(stream, start=1):
        now = time.perf_counter()
        stage["total_ms"].append((now - prev) * 1000)
        prev = now
        if r.boxes is not None and getattr(r.boxes, "id", None) is not None:
            ids = r.boxes.id.cpu().numpy().astype(int)
            xyxy = r.boxes.xyxy.cpu().numpy()
            clss = r.boxes.cls.cpu().numpy().astype(int)
            for tid, box, cls in zip(ids, xyxy, clss):
                if int(cls) not in COCO_NAMES:
                    continue
                x1, y1, x2, y2 = box.tolist()
                events.append((fi, int(tid), int(cls), x1, y1, x2, y2))
        elif r.boxes is None or len(r.boxes) == 0:
            dropped += 0  # empty detections ≠ dropped frame
    total_s = time.perf_counter() - t_run0
    fps = len(files) / max(total_s, 1e-9)
    arr = np.array(stage["total_ms"]) if stage["total_ms"] else np.array([0.0])
    return {
        "n_frames": len(files),
        "total_s": total_s,
        "fps": fps,
        "p50_ms": float(np.percentile(arr, 50)),
        "p95_ms": float(np.percentile(arr, 95)),
        "mean_ms": float(arr.mean()),
        "max_ms": float(arr.max()),
        "n_events": len(events),
        "n_tracks": len(set(e[1] for e in events)) if events else 0,
        "events": events,
        "per_frame_ms": arr,
    }

print("bench helper ready")

In [ ]:
# --- 3 прогони БЕЗ прогріву ---
runs = []
for i in range(BENCH_RUNS):
    print(f"\n=== RUN {i+1}/{BENCH_RUNS} (no warmup) ===")
    # новий model щоб не тримати гарячий стан як «прогрів» між логічними прогонами
    m = YOLO(DETECTOR_WEIGHTS)
    res = run_pipeline_timed(m, SEQ_PATH, tracker=TRACKER_CFG, imgsz=IMGSZ, conf=CONF, iou=IOU)
    runs.append(res)
    print(f"  FPS={res['fps']:.2f}  p50={res['p50_ms']:.1f}ms  p95={res['p95_ms']:.1f}ms  tracks={res['n_tracks']}")

bench_df = pd.DataFrame([
    {
        "run": i + 1,
        "FPS": round(r["fps"], 2),
        "p50_ms": round(r["p50_ms"], 2),
        "p95_ms": round(r["p95_ms"], 2),
        "mean_ms": round(r["mean_ms"], 2),
        "max_ms": round(r["max_ms"], 2),
        "n_tracks": r["n_tracks"],
        "n_events": r["n_events"],
    }
    for i, r in enumerate(runs)
])
print("\n=== 3 runs summary ===")
display(bench_df)
print(f"Mean FPS: {bench_df['FPS'].mean():.2f}")
print(f"Mean p50: {bench_df['p50_ms'].mean():.1f} ms | Mean p95: {bench_df['p95_ms'].mean():.1f} ms")

In [ ]:
# Вузьке місце vs бюджет кадру
mean_p95 = float(bench_df["p95_ms"].mean())
mean_p50 = float(bench_df["p50_ms"].mean())
mean_fps = float(bench_df["FPS"].mean())

print("=== Bottleneck vs frame budget ===")
print(f"Target example: 15 FPS → budget {FRAME_BUDGET_MS:.1f} ms/frame")
print(f"Measured p50={mean_p50:.1f} ms, p95={mean_p95:.1f} ms, FPS≈{mean_fps:.1f}")
if mean_p95 > FRAME_BUDGET_MS:
    print(f"⚠ p95 ({mean_p95:.1f} ms) > budget ({FRAME_BUDGET_MS:.1f} ms) → ризик зриву real-time @15FPS")
else:
    print(f"✓ p95 ({mean_p95:.1f} ms) ≤ budget ({FRAME_BUDGET_MS:.1f} ms) @15FPS")

# Основна стадія = detect+track (ultralytics об'єднує); вузьке місце — інференс детектора на GPU
print("Вузьке місце: YOLO detect+ByteTrack на кадр (єдина виміряна стадія end-to-end).")

In [ ]:
# --- Інтервенція: imgsz 640 → 480 (швидше) — до/після ---
print("=== INTERVENTION: imgsz 640 → 480 ===")
m_before = YOLO(DETECTOR_WEIGHTS)
before = run_pipeline_timed(m_before, SEQ_PATH, tracker=TRACKER_CFG, imgsz=640, conf=CONF, iou=IOU)
m_after = YOLO(DETECTOR_WEIGHTS)
after = run_pipeline_timed(m_after, SEQ_PATH, tracker=TRACKER_CFG, imgsz=480, conf=CONF, iou=IOU)

inter_df = pd.DataFrame([
    {"setting": "imgsz=640 (before)", "FPS": round(before["fps"], 2),
     "p50_ms": round(before["p50_ms"], 2), "p95_ms": round(before["p95_ms"], 2),
     "n_tracks": before["n_tracks"]},
    {"setting": "imgsz=480 (after)", "FPS": round(after["fps"], 2),
     "p50_ms": round(after["p50_ms"], 2), "p95_ms": round(after["p95_ms"], 2),
     "n_tracks": after["n_tracks"]},
])
display(inter_df)
print(f"FPS change: {before['fps']:.1f} → {after['fps']:.1f} ({(after['fps']-before['fps'])/max(before['fps'],1e-9)*100:+.1f}%)")

In [ ]:
# --- Класи відмов (мінімум 2) на основі events останнього повного прогону ---
ev = runs[-1]["events"]  # list of (frame, tid, cls, x1,y1,x2,y2)

# 1) Short tracks (ID flicker / fragment): length == 1 frame
from collections import Counter
track_len = Counter(e[1] for e in ev)
short_ids = [tid for tid, ln in track_len.items() if ln == 1]
short_frames = sorted({e[0] for e in ev if e[1] in short_ids})[:10]

# 2) Frames with zero tracks (missed detections on frame)
frames_with = set(e[0] for e in ev)
empty_frames = [i for i in range(1, n_frames + 1) if i not in frames_with]

print("=== Failure classes ===")
print(f"1) Short tracks (len=1): count={len(short_ids)}  example frames={short_frames[:5]}")
print(f"2) Empty frames (no tracks): count={len(empty_frames)}  example frames={empty_frames[:5]}")

# optional 3) ID switches proxy: same spatial region different ids across adjacent frames — skip if no GT
print("\nIDF1 / IDSW: GT MOT annotations not loaded for demo sequence → N/A.")
print("When real VisDrone-MOT annotations are present, compute with motmetrics (IDF1, IDSW).")

In [ ]:
# Зведена таблиця для звіту 05
summary = {
    "sequence": SEQ_PATH.name,
    "n_frames": n_frames,
    "runs": BENCH_RUNS,
    "mean_FPS": round(mean_fps, 2),
    "mean_p50_ms": round(mean_p50, 2),
    "mean_p95_ms": round(mean_p95, 2),
    "frame_budget_ms_15fps": round(FRAME_BUDGET_MS, 1),
    "intervention": "imgsz 640→480",
    "FPS_before": round(before["fps"], 2),
    "FPS_after": round(after["fps"], 2),
    "short_tracks": len(short_ids),
    "empty_frames": len(empty_frames),
    "IDF1": "N/A (no MOT GT)",
    "IDSW": "N/A (no MOT GT)",
}
print("=== BENCH SUMMARY FOR REPORT ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

pd.DataFrame([summary]).to_csv(PIPE_OUT / "bench_summary.csv", index=False)
print("Saved", PIPE_OUT / "bench_summary.csv")

### Висновок (шаблон 5–8 речень для звіту)

1. У ДЗ3 йдуть **detector (YOLO boxes)** + **ByteTrack**.  
2. Кроки 1–3: detector обрано за mAP і GPU; найгірша деградація — **noise**.  
3. Пайплайн видає MOT-лог і події з `track_id` у координатах повного кадру.  
4. Бенчмарк (3 прогони): FPS / p50 / p95 — підставити цифри з таблиці.  
5. Вузьке місце — end-to-end detect+track; порівняти p95 з бюджетом кадру.  
6. Інтервенція imgsz 640→480: FPS до/після — підставити.  
7. Класи відмов: short tracks, empty frames (числа з виводу).  
8. Не перевірено: справжній MOT GT (IDF1/IDSW), RTSP, TensorRT, комбінації деградацій на відео.